# Table 1: International diffusion of scientific ideas

This notebook fits the weighted logistic regression to predict the fraction of ideas diffused from an origin country to a destination country, and computes the variance inflation factors.

It reads the processed data in the data folder (`path2data`) and can be run on its own.

In [1]:
import os
import json
import gzip
import itertools
from itertools import chain
import pandas as pd
import numpy as np
import networkx as nx

import scipy.sparse as spsparse
import scipy.stats as spstats

from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as mtransforms
import seaborn as sns

from vizutils import pvalue2stars, table_row, make_multinomial_latex_table, logistic_coefficient_plot

import pyscisci.all as pyscisci

# set this path to where the OpenAlex database is stored
path2openalex = '/Users/hgt6rn/Documents/DataSets/OpenAlex'

# set this path to where the processed data (Source Data) is stored:
path2sourcedata = "Source Data"
# when the notebook is run from the code folder, the Source Data folder is one level up
if not os.path.exists(path2sourcedata):
    path2sourcedata = os.path.join("..", "Source Data")

# the processed input data are in the data folder, next to the Source Data folder
path2data = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'data')


Please install python-Levenshtein to take full advantage of the NLP tools. 
 pip install python-Levenshtein
Please install sparse_dot_topn to take full advantage of the NLP tools. 
 pip install sparse_dot_topn


/Users/psp2nq/anaconda3/lib/python3.11/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

namedict = {'const':'Intercept', 
            'oTopCumuFrac':'Source top journal fraction', 'tTopCumuFrac':'Target top journal fraction',
            'oPop':'Source population','tPop':'Target population',
             'oGDP_PCAP':'Source GDP per capita', 'tGDP_PCAP':'Target GDP per capita',
            'oTopFrac':'Source top journal fraction', 'tTopFrac':'Target top journal fraction',
            'PhysicalDistance': 'Physical distance', 'tradeflow_baci':'Trade volume', 
            'FieldDistance':'Topic distance', 'SameContinent':'Same continent',
            'CollabStrength':'Collaboration strength',
            'CosineNormStrength':'Normed collaboration strength',
            'comlang_off':'Same official language', 
            'DemocDis': 'Democracy distance','oDemoc':'Source democracy', 'tDemoc':'Target democracy',
           'oGNI_PCAP':'Source GNI per capita', 'tGNI_PCAP':'Target GNI per capita',
           'GNI_PCAP':'GNI per capita', 'GNI':'GNI', 'GDP_PCAP':'GDP per capita',
           'Pop':'Population', 'NumPub':'Number of publications','Agreements':'Bilateral research agreement',
           'RND_per':'Percent of GDP for R&D', "FractionNationalAuthors":"Fraction national authors"}


# Weighted Logistic Regression Model for Idea Diffusion

In [3]:
panel_data = pd.read_csv(os.path.join(path2data, 'diffusion_panel_data_1990_2017.csv.gz'))



network_features = [ 'Pedge', 'Nedge'] 
#network_features = [ 'Pedge', 'Nedge', 'SameComm'] 
#network_features = [ 'Pedge', 'Nedge',  'pr_o', 'pr_d'] 
network_features = [ 'Pedge', 'Nedge', 'SameComm', 'pr_o', 'pr_d'] 

Model_features = [
#'Pop_o','Pop_d', 
'GDP_PCAP_o','GDP_PCAP_d',
'NumPub_o','NumPub_d', 
'NormNcites',  
'FieldDistance', 
'dist', 
'comlang_off'
] + network_features


X = panel_data[Model_features + [ 'Norigin', 'Fideas']].dropna().copy()
Y = X['Fideas']
freq_weights = X['Norigin'].values
X = sm.add_constant(X[Model_features])

for c in Model_features:
    if X[c].nunique() > 2:
        X[c] = (X[c] - X[c].mean())/X[c].std()

vif_df = pd.DataFrame()
vif_df["Variable"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

vif_df['Variable']=vif_df['Variable'].replace(namedict)

#model=sm.GLM(Y, X, family=sm.families.Binomial(sm.families.links.logit())).fit()
model=sm.GLM(Y, X, family=sm.families.Binomial(sm.families.links.logit()), freq_weights=freq_weights).fit()

predictions = model.predict(X) 
print(model.pseudo_rsquared(kind='mcf'))
params = model.params
conf = model.conf_int()
conf['Odds Ratio'] = params
conf.columns = ['5%', '95%', 'Odds Ratio']
print(np.exp(conf))

model.summary()

0.4248119341799661
                     5%       95%  Odds Ratio
const          0.254320  0.262404    0.258330
GDP_PCAP_o     1.319744  1.343300    1.331470
GDP_PCAP_d     1.083653  1.093667    1.088649
NumPub_o       0.730610  0.746141    0.738335
NumPub_d       4.741752  4.796227    4.768912
NormNcites     1.054395  1.058477    1.056434
FieldDistance  0.714497  0.721512    0.717996
dist           1.010699  1.019164    1.014923
comlang_off    0.989790  1.010325    1.000005
Pedge          1.126826  1.155277    1.140963
Nedge          0.952675  0.969649    0.961125
SameComm       1.071280  1.095541    1.083343
pr_o           1.007300  1.015541    1.011412
pr_d           1.039934  1.048757    1.044336


/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                 Fideas   No. Observations:                12032
Model:                            GLM   Df Residuals:                  2814066
Model Family:                Binomial   Df Model:                           13
Link Function:                  logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -7.9875e+05
Date:                Wed, 04 Nov 2026   Deviance:                       39890.
Time:                        16:09:34   Pearson chi2:                 4.34e+04
No. Iterations:                     6   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -1.3535      0.008   -169.549      0.000      -1.369      -1.338
GDP_PCAP_o        0.2863      0.005     63.434      0.000       0.277       0.295
GDP_PCAP_d        0.0849      0.002     36.196      0.000       0.080       0.090
NumPub_o         -0.3034      0.005    -56.533      0.000      -0.314      -0.293
NumPub_d          1.5621      0.003    536.070      0.000       1.556       1.568
NormNcites        0.0549      0.001     55.694      0.000       0.053       0.057
FieldDistance    -0.3313      0.002   -132.926      0.000      -0.336      -0.326
dist              0.0148      0.002      6.962      0.000       0.011       0.019
comlang_off    5.065e-06      0.005      0.001      0.999      -0.010       0.010
Pedge             0.1319      0.006     20.730      0.000       0.119       0.144
Nedge            -0.0397      0.005     -8.801      0.000      -0.048      -0.031
SameComm          0.0801      0.006     14.012      0.000       0.069       0.091
pr_o              0.0113      0.002      5.459      0.000       0.007       0.015
pr_d              0.0434      0.002     20.130      0.000       0.039       0.048
=================================================================================
"""

## Source Data for Table 1

The values reported in the table, written to `Source Data/SourceData_5_Table1.xlsx`.

In [4]:
# the Source Data helpers (sourcedatautils.py, in the code folder like vizutils.py)
from sourcedatautils import write_source_data

title = 'Table 1: International diffusion of scientific ideas.'
description = ('Coefficients of the weighted logistic regression models (1)-(4) predicting the fraction of ideas '
               'diffused from an origin country to a destination country (Fideas), weighted by the number of ideas '
               'originating from the origin country (Norigin). Continuous variables are standardized. '
               'Pseudo R2 is McFadden pseudo R2.')

table1_names = {'const':'Intercept', 'GDP_PCAP_o':'log GDP per capita origin', 'GDP_PCAP_d':'log GDP per capita destination',
                'NumPub_o':'log Number of publications origin', 'NumPub_d':'log Number of publications destination',
                'NormNcites':'Fraction of citations to origin', 'FieldDistance':'Topic distance',
                'dist':'log Physical distance', 'comlang_off':'Common language',
                'Pedge':'Positive citation preference', 'Nedge':'Negative citation preference',
                'SameComm':'Same network community', 'pr_o':'Network centrality origin',
                'pr_d':'Network centrality destination'}

# the nested models in Table 1; Model (4) is the model fitted above
model_sets = [['GDP_PCAP_o', 'GDP_PCAP_d', 'NumPub_o', 'NumPub_d', 'NormNcites'],
              ['FieldDistance', 'dist', 'comlang_off'],
              ['Pedge', 'Nedge'],
              ['SameComm', 'pr_o', 'pr_d']]

coefs = []
fits = []
table1_models = []
for imodel in range(len(model_sets)):
    features = list(chain.from_iterable(model_sets[:(imodel+1)]))

    mX = panel_data[features + ['Norigin', 'Fideas']].dropna().copy()
    mY = mX['Fideas']
    mweights = mX['Norigin'].values
    mX = sm.add_constant(mX[features])
    for c in features:
        if mX[c].nunique() > 2:
            mX[c] = (mX[c] - mX[c].mean())/mX[c].std()

    mmodel = sm.GLM(mY, mX, family=sm.families.Binomial(sm.families.links.logit()), freq_weights=mweights).fit()

    table1_models.append(mmodel)

    mconf = mmodel.conf_int()
    df = pd.DataFrame({'Model':imodel + 1, 'Variable':[table1_names[v] for v in mmodel.params.index],
                       'Coefficient':mmodel.params.values, 'CI lower (95%)':mconf[0].values,
                       'CI upper (95%)':mconf[1].values, 'SE':mmodel.bse.values, 'p-value':mmodel.pvalues.values})
    coefs.append(df)
    fits.append([imodel + 1, int(mmodel.nobs), mmodel.pseudo_rsquared(kind='mcf'), mmodel.llf])

coefs = pd.concat(coefs, ignore_index=True)
fits = pd.DataFrame(fits, columns=['Model', 'Observations', 'Pseudo R2', 'Log likelihood'])

# the model input, before standardization
panel_columns = ['Origin', 'Country', 'Norigin', 'Fideas'] + list(chain.from_iterable(model_sets))
model_input = panel_data[panel_columns].rename(columns={'Origin':'Origin country', 'Country':'Destination country'})

tables = [('Coefficients', 'Coefficients of each model.', coefs),
          ('Model fit', 'Observations and goodness of fit of each model.', fits),
          ('Model input', 'Country-pair panel used in the models (variables before standardization).', model_input)]

write_source_data('SourceData_5_Table1.xlsx', title, description, tables)


/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(
/Users/psp2nq/anaconda3/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The logit link alias is deprecated. Use Logit instead. The logit link alias will be removed after the 0.15.0 release.
  warnings.warn(


wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_5_Table1.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt


## Statistical tests reported in the main text

Likelihood-ratio tests between the nested models and Wald tests of equal coefficients.

In [5]:
# likelihood-ratio tests between the nested models of Table 1
for i in range(1, len(table1_models)):
    restricted, full = table1_models[i - 1], table1_models[i]
    lr = 2*(full.llf - restricted.llf)
    dof = int(full.df_model - restricted.df_model)
    print("Model {} vs Model {}: chi2({}) = {:.1f}, p = {:.3g}".format(i + 1, i, dof, lr, spstats.chi2.sf(lr, dof)))

# Wald tests of equal coefficients (two-sided)
for imodel, (a, b) in [(1, ('GDP_PCAP_o', 'GDP_PCAP_d')), (4, ('pr_d', 'pr_o'))]:
    m = table1_models[imodel - 1]
    test = m.t_test('{} - {} = 0'.format(a, b))
    print("Model {}: {} - {} = {:.3f}, z = {:.1f}, p = {:.3g}".format(imodel, a, b, test.effect[0],
          test.tvalue.item(), test.pvalue.item()))

# the positive citation preference before and after adding the community variables
for imodel in [3, 4]:
    m = table1_models[imodel - 1]
    ci = m.conf_int().loc['Pedge']
    print("Model {}: Pedge beta = {:.3f}, 95% CI [{:.3f}, {:.3f}], p = {:.3g}".format(imodel, m.params['Pedge'],
          ci[0], ci[1], m.pvalues['Pedge']))


Model 2 vs Model 1: chi2(3) = 155680.3, p = 0
Model 3 vs Model 2: chi2(2) = 1040.3, p = 1.29e-226
Model 4 vs Model 3: chi2(3) = 640.7, p = 1.52e-138
Model 1: GDP_PCAP_o - GDP_PCAP_d = 0.161, z = 39.2, p = 0
Model 4: pr_d - pr_o = 0.032, z = 10.8, p = 2.22e-27
Model 3: Pedge beta = 0.167, 95% CI [0.154, 0.179], p = 1.24e-159
Model 4: Pedge beta = 0.132, 95% CI [0.119, 0.144], p = 1.85e-95


## Table 1 (LaTeX)

In [6]:
from vizutils import make_glm_latex_table

# the LaTeX tables are saved in the tables folder
path2tables = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'tables')
os.makedirs(path2tables, exist_ok=True)

caption = ("\\textbf{International diffusion of scientific ideas.}\n"
           "Model coefficients for a series of weighted logistic regression models to predict the probability of ideas diffusing "
           "from an origin country to a destination country. The dependent variable is the observed fraction of ideas diffused "
           "from the origin country to the destination country, and the weights correspond to the total number of ideas "
           "originating from the origin country. Model coefficients (log-odds) are reported with 95\\% confidence intervals in "
           "parentheses, followed by the standard error (S.E.) and the exact $p$-value. "
           "McFadden's pseudo-$R^2$ is reported. $p$-values are from two-sided Wald $z$-tests of each coefficient, without "
           "adjustment for multiple comparisons. The F statistic is a Wald test that all coefficients except the intercept "
           "are zero. Source data are provided as a Source Data file.")

make_glm_latex_table(table1_models, model_sets, table1_names, caption_text=caption,
                     dep_var_text="Fraction of ideas diffused from origin to destination country.",
                     label="table:idea_diffusion", fname=os.path.join(path2tables, "Table1.tex"))

\begin{table}[H] \centering 
    \small
  \caption{\textbf{International diffusion of scientific ideas.}
Model coefficients for a series of weighted logistic regression models to predict the probability of ideas diffusing from an origin country to a destination country. The dependent variable is the observed fraction of ideas diffused from the origin country to the destination country, and the weights correspond to the total number of ideas originating from the origin country. Model coefficients (log-odds) are reported with 95\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. McFadden's pseudo-$R^2$ is reported. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The F statistic is a Wald test that all coefficients except the intercept are zero. Source data are provided as a Source Data file.} 
\label{table:idea_diffusion}
\begin{adjustbox}{max width=\textwidth} 
\begin{tabula

"\\begin{table}[H] \\centering \n    \\small\n  \\caption{\\textbf{International diffusion of scientific ideas.}\nModel coefficients for a series of weighted logistic regression models to predict the probability of ideas diffusing from an origin country to a destination country. The dependent variable is the observed fraction of ideas diffused from the origin country to the destination country, and the weights correspond to the total number of ideas originating from the origin country. Model coefficients (log-odds) are reported with 95\\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. McFadden's pseudo-$R^2$ is reported. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The F statistic is a Wald test that all coefficients except the intercept are zero. Source data are provided as a Source Data file.} \n\\label{table:idea_diffusion}\n\\begin{adjustbox}{max width=\\textwidth}